# EDA — organic_ratio cleaned target table

Загружает `data/train/targets_train_clean.parquet` и смотрит:
1. shape, dtypes, NaN map
2. распределение target (`organic_share`)
3. распределение `total_installs` (масса когорт)
4. target по платформам и топ-странам
5. корреляции с target (топ-20)
6. multicollinearity (corr matrix топ-фичей)

Запускать из корня проекта (`cd organic_ratio && jupyter lab`).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "organic_ratio").is_dir():
            return p
    raise RuntimeError(f"organic_ratio src not found upward from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)
print("project root:", PROJECT_ROOT)
print("cwd:", Path.cwd())

from organic_ratio.utils.config import load_config

sns.set_context("notebook")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
cfg = load_config()
out_cfg = cfg.datasets.targets

train_path = Path(out_cfg.train_dir) / out_cfg.train_clean_filename
test_path = Path(out_cfg.test_dir) / out_cfg.test_clean_filename

train = pl.read_parquet(train_path).to_pandas()
test = pl.read_parquet(test_path).to_pandas()

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
train.head()

## 1. Schema + NaN map

In [ ]:
n = len(train)
info = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(),
    "n_null": train.isna().sum(),
    "pct_null": (train.isna().sum() / n * 100).round(2),
})
info.sort_values("pct_null", ascending=False).head(40)

## 2. Distribution of `organic_share`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(train["organic_share"], bins=50, edgecolor="black")
axes[0].set_title(f"train  (n={len(train):,})")
axes[0].set_xlabel("organic_share")
axes[0].axvline(train["organic_share"].mean(), color="r", ls="--", label=f"mean={train['organic_share'].mean():.3f}")
axes[0].axvline(train["organic_share"].median(), color="g", ls="--", label=f"median={train['organic_share'].median():.3f}")
axes[0].legend()

axes[1].hist(test["organic_share"], bins=50, edgecolor="black", color="orange")
axes[1].set_title(f"test  (n={len(test):,})")
axes[1].set_xlabel("organic_share")
axes[1].axvline(test["organic_share"].mean(), color="r", ls="--", label=f"mean={test['organic_share'].mean():.3f}")
axes[1].axvline(test["organic_share"].median(), color="g", ls="--", label=f"median={test['organic_share'].median():.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# logit(y) — если хвосты у 0/1 тонкие, логит ляжет нормально и можно делать линейный регрессор
eps = 1e-3
y_clip = train["organic_share"].clip(eps, 1 - eps)
logit_y = np.log(y_clip / (1 - y_clip))

plt.figure(figsize=(8, 4))
plt.hist(logit_y, bins=60, edgecolor="black")
plt.title("logit(organic_share) — train")
plt.xlabel("logit")
plt.axvline(0, color="r", ls="--", alpha=0.5, label="share=0.5")
plt.legend()
plt.show()

## 3. `total_installs` — масса когорт

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(train["total_installs"], bins=80, edgecolor="black")
axes[0].set_title("total_installs (linear)")
axes[0].set_xlabel("installs")

axes[1].hist(np.log10(train["total_installs"]), bins=60, edgecolor="black")
axes[1].set_title("log10(total_installs)")
axes[1].set_xlabel("log10 installs")

plt.tight_layout()
plt.show()

print(train["total_installs"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))

In [ ]:
# проверка: маленькие когорты должны давать более «шумный» (расплывчатый) target
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(np.log10(train["total_installs"]), train["organic_share"], alpha=0.15, s=8)
ax.set_xlabel("log10(total_installs)")
ax.set_ylabel("organic_share")
ax.set_title("organic_share vs cohort size")
plt.show()

## 4. organic_share по платформам и топ-странам

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=train, x="platform", y="organic_share", ax=ax)
ax.set_title("organic_share by platform — train")
plt.show()

print(train.groupby("platform")["organic_share"].agg(["count", "mean", "std", "median"]))

In [ ]:
country_stats = (
    train.groupby("country_code")
    .agg(n_cohorts=("organic_share", "size"),
         total_installs=("total_installs", "sum"),
         share_mean=("organic_share", "mean"),
         share_std=("organic_share", "std"))
    .sort_values("total_installs", ascending=False)
)
top20 = country_stats.head(20).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top20["country_code"], top20["share_mean"], yerr=top20["share_std"], capsize=3)
ax.set_title("organic_share by top-20 countries (by install volume)")
ax.set_ylabel("organic_share")
ax.axhline(train["organic_share"].mean(), color="r", ls="--", alpha=0.6, label="global mean")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

top20

## 5. Корреляция фичей с target

In [ ]:
TARGET = "organic_share"
DROP = {"organic_share", "organic_installs", "total_installs",
        "country_code", "platform", "install_date"}

num_cols = [c for c in train.select_dtypes(include=[np.number]).columns if c not in DROP]
print(f"numeric features for correlation: {len(num_cols)}")

# Pearson + Spearman (Spearman устойчив к выбросам и нелинейностям)
pearson = train[num_cols + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
spearman = train[num_cols + [TARGET]].corr(method="spearman", numeric_only=True)[TARGET].drop(TARGET)

corr_df = pd.DataFrame({
    "pearson": pearson,
    "spearman": spearman,
    "abs_spearman": spearman.abs(),
}).sort_values("abs_spearman", ascending=False)

corr_df.head(25)

In [ ]:
top = corr_df.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top["spearman"])
ax.axvline(0, color="k", lw=0.5)
ax.set_title("Top-20 features by |spearman corr| with organic_share")
ax.set_xlabel("spearman corr")
plt.tight_layout()
plt.show()

## 6. Multicollinearity (топ-фичи между собой)

In [ ]:
TOP_N = 20
top_features = corr_df.head(TOP_N).index.tolist()

mc = train[top_features].corr(method="spearman")

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(mc, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.75}, ax=ax)
ax.set_title(f"Multicollinearity heatmap (top-{TOP_N} features by |corr| with target)")
plt.tight_layout()
plt.show()

In [ ]:
# топ-30 пар фичей с самой сильной корреляцией между собой (≥ 0.8 — кандидаты на дроп)
abs_mc = mc.abs()
tri = abs_mc.where(np.triu(np.ones(abs_mc.shape), k=1).astype(bool))
pairs = (
    tri.stack()
    .sort_values(ascending=False)
    .head(30)
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b", 0: "abs_spearman"})
)
pairs

## 7. Динамика во времени

In [ ]:
daily = (
    train.assign(install_date=pd.to_datetime(train["install_date"]))
    .groupby(["install_date", "platform"])
    .agg(share=("organic_share", "mean"),
         n=("organic_share", "size"),
         installs=("total_installs", "sum"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
for p, sub in daily.groupby("platform"):
    ax.plot(sub["install_date"], sub["share"], label=p, alpha=0.85)
ax.set_title("mean organic_share by day, by platform (train)")
ax.set_ylabel("organic_share")
ax.legend()
plt.tight_layout()
plt.show()

## Что искать в выводе

- **Schema/NaN**: сколько пропусков в `cpi/ctr/cvr/cpm` (когорты без paid spend). Это сигнал, не шум — в модели можно или заполнять 0, или явно кодировать `has_paid_spend` бинарным флагом.
- **organic_share hist**: должен быть унимодальный или умеренно-скошенный, без пиков на 0/1.
- **logit(y)**: если близко к нормальному → линейная регрессия в logit-пространстве будет работать; если хвосты тяжёлые → нужно Beta-likelihood (PyMC).
- **scatter share vs size**: разброс должен сужаться с ростом cohort_size (так и должно быть — большая когорта = устойчивая оценка).
- **by-platform / by-country**: видим ли структурные различия — это аргумент за иерархическую модель с partial pooling.
- **Top-corr features**: интуитивные ли это предикторы? UA-метрики (`spend`, `clicks`, `cpi`) должны коррелировать отрицательно с organic_share, retention-метрики — менее очевидно.
- **Multicollinearity**: `spend ↔ impressions ↔ clicks ↔ installs_spend` будут на 0.9+ — для линейных моделей нужно выбрать одну из них либо PCA / VIF.
- **Time series**: должны быть видны спады/всплески в дни компанийных событий (релизы, акции).